# Session 32: Logistic Regression Classification Baseline
**Week 3 Baseline Classifier for At-Risk Target Prediction**

In this notebook, we shift from regression tasks to binary classification. We construct an "at-risk" target variable ($G3 < 10$) and fit a scaled Logistic Regression baseline model using a scikit-learn `Pipeline`.

In [6]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, classification_report, confusion_matrix
)

# Resolve project directories
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / ".git").exists():
    for parent in PROJECT_ROOT.parents:
        if (parent / ".git").exists():
            PROJECT_ROOT = parent
            break

DATA_DIRECTORY = PROJECT_ROOT / "data"
REPORTS_DIRECTORY = PROJECT_ROOT / "reports"
TABLES_DIRECTORY = REPORTS_DIRECTORY / "tables"

TABLES_DIRECTORY.mkdir(parents=True, exist_ok=True)
print("Project Root:", PROJECT_ROOT)

Project Root: /home/nikhil/Desktop/VSCode/GSSRP/student-performance-prediction-ml


In [7]:
def load_and_transform_data():
    # Attempt to load the same feature splits used in regression
    # If loading raw processed table, split using the same random state
    from sklearn.model_selection import train_test_split
    
    candidates = list((DATA_DIRECTORY / "processed").rglob("*.parquet")) + list((DATA_DIRECTORY / "processed").rglob("*.csv"))
    for path in candidates:
        if any(term in path.name.lower() for term in ["comparison", "prediction", "result"]):
            continue
        try:
            table = pd.read_parquet(path) if path.suffix == ".parquet" else pd.read_csv(path)
            if "G3" in table.columns:
                X = table.drop(columns=["G3"]).copy()
                X = pd.get_dummies(X, drop_first=True, dtype=float)
                y_reg = table["G3"]
                
                # Critical step: Convert regression target to binary classification target
                # At-risk (1) if G3 < 10, otherwise Pass (0)
                y_clf = (y_reg < 10).astype(int)
                
                Xtr, Xte, yctr, ycte = train_test_split(
                    X, y_clf, test_size=0.20, random_state=42
                )
                return Xtr, Xte, yctr, ycte
        except Exception as e:
            continue
    raise FileNotFoundError("Could not locate source data splits.")

Xtr_f, Xte_f, yctr, ycte = load_and_transform_data()
print(f"Train shapes: X={Xtr_f.shape}, y={yctr.shape}")
print(f"At-risk prevalence in training: {yctr.mean():.2%}")

Train shapes: X=(316, 41), y=(316,)
At-risk prevalence in training: 32.59%


In [8]:
# Instantiate scaled Logistic Regression pipeline
clf = make_pipeline(
    StandardScaler(), 
    LogisticRegression(max_iter=1000, random_state=42)
)

# Fit classifier on binarized training data
clf.fit(Xtr_f, yctr)

# Generate class and probability predictions
predictions = clf.predict(Xte_f)
probabilities = clf.predict_proba(Xte_f)[:, 1]

# Compute standard evaluation metrics
accuracy = accuracy_score(ycte, predictions)
precision = precision_score(ycte, predictions, zero_division=0)
recall = recall_score(ycte, predictions, zero_division=0)
f1 = f1_score(ycte, predictions, zero_division=0)
roc_auc = roc_auc_score(ycte, probabilities)

print("Baseline Logistic Regression Performance:")
print("-" * 40)
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"ROC AUC:   {roc_auc:.4f}\n")

print("Classification Report:")
print(classification_report(ycte, predictions))

Baseline Logistic Regression Performance:
----------------------------------------
Accuracy:  0.8987
Precision: 0.8276
Recall:    0.8889
F1 Score:  0.8571
ROC AUC:   0.9736

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.90      0.92        52
           1       0.83      0.89      0.86        27

    accuracy                           0.90        79
   macro avg       0.88      0.90      0.89        79
weighted avg       0.90      0.90      0.90        79



In [9]:
classification_path = TABLES_DIRECTORY / "classification_model_comparison.csv"

# Construct the output tracking row matching target definitions
log_row = pd.DataFrame([{
    "Session": 32,
    "Week": 3,
    "Task Type": "Classification",
    "Scenario": "Full-information",
    "Feature Set": "X_full",
    "Target": "At-Risk (G3 < 10)",
    "Model": "Logistic Regression",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1 Score": f1,
    "ROC AUC": roc_auc
}])

if classification_path.exists():
    comparison_table = pd.read_csv(classification_path)
    comparison_table = comparison_table[comparison_table["Session"] != 32].copy()
    comparison_table = pd.concat([comparison_table, log_row], ignore_index=True)
else:
    comparison_table = log_row.copy()

comparison_table.to_csv(classification_path, index=False)
print("Updated classification comparison table saved at:")
print(classification_path)
display(comparison_table.round(4))

Updated classification comparison table saved at:
/home/nikhil/Desktop/VSCode/GSSRP/student-performance-prediction-ml/reports/tables/classification_model_comparison.csv


,Session,Week,Task Type,Scenario,Feature Set,Target,Model,Accuracy,Precision,Recall,F1 Score,ROC AUC
0,32,3,Classification,Full-information,X_full,At-Risk (G3 < 10),Logistic Regression,0.8987,0.8276,0.8889,0.8571,0.9736


In [10]:
# Validate scaling step and architecture parameters
assert isinstance(clf.steps[0][1], StandardScaler), "Pipeline step 0 must be StandardScaler!"
assert isinstance(clf.steps[1][1], LogisticRegression), "Pipeline step 1 must be LogisticRegression!"
assert clf.steps[1][1].max_iter == 1000, "max_iter must be set to 1000!"

# Verify data shapes and types
assert set(np.unique(yctr)).issubset({0, 1}), "Target classes must be strictly binary 0 and 1!"
assert len(predictions) == len(ycte), "Prediction size mismatch!"

print("SESSION 32 NOTEBOOK SECTION COMPLETED SUCCESSFULLY")

SESSION 32 NOTEBOOK SECTION COMPLETED SUCCESSFULLY


## Session 33: KNN, SVM, and Naive Bayes Classification
This section extends the classification-model notebook with three additional classifier families:
1. K-Nearest Neighbors (KNN)
2. Support Vector Machine (SVM)
3. Gaussian Naive Bayes (NB)

All three classifiers use the same training and test observations. KNN and SVM require scaling because their results depend directly on feature distances or margins. Gaussian Naive Bayes is included in the same pipeline to preserve a consistent modeling workflow. The required output is an updated classification table containing KNN, SVM, and NB rows.

In [11]:
# Session 33 imports and prerequisite validation
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

# Ensure baseline variables from Session 32 exist in kernel memory
required_session33_objects = ["Xtr_f", "Xte_f", "yctr", "ycte"]
missing_objects = [obj for obj in required_session33_objects if obj not in globals()]

if missing_objects:
    raise NameError(f"Missing required preprocessing splits: {missing_objects}. Run Session 32 cells first.")

assert len(Xtr_f) == len(yctr), "Xtr_f and yctr must contain the same number of rows."
assert len(Xte_f) == len(ycte), "Xte_f and ycte must contain the same number of rows."

print("Session 33 prerequisites verified successfully.")

Session 33 prerequisites verified successfully.


### Target Class Mapping Context
The notebook utilizes the project target definitions:
* `0` = At-risk student (Final grade below 10)
* `1` = Successful student (Final grade 10 or higher)

Standard metrics below treat class 1 (successful) as the positive class. To gain deep visibility into our early warning task, additional tracking blocks isolate performance treating class 0 (at-risk) as the primary outcome.

In [12]:
def evaluate_session33_classifier(y_true, y_pred, y_probability):
    """
    Return standard and at-risk classification metrics.
    Standard precision, recall, and F1 treat class 1 as positive.
    At-risk precision, recall, and F1 treat class 0 as positive.
    """
    unique_classes = np.unique(np.asarray(y_true).ravel())
    roc_auc = roc_auc_score(y_true, y_probability) if len(unique_classes) == 2 else np.nan
    
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc,
        "at_risk_precision": precision_score(y_true, y_pred, pos_label=0, zero_division=0),
        "at_risk_recall": recall_score(y_true, y_pred, pos_label=0, zero_division=0),
        "at_risk_f1": f1_score(y_true, y_pred, pos_label=0, zero_division=0),
    }

print("Session 33 classification evaluator is ready.")

Session 33 classification evaluator is ready.


In [13]:
# Establish target structural estimators
session33_estimators = [
    ("KNN", "K-Nearest Neighbors", "Instance-based", KNeighborsClassifier()),
    ("SVM", "Support Vector Machine", "Maximum-margin", SVC(probability=True, random_state=42)),
    ("NB", "Gaussian Naive Bayes", "Probabilistic", GaussianNB())
]

session33_models = {}
session33_result_rows = []

for model_code, full_model_name, model_family, estimator in session33_estimators:
    # Wrap standard scaler around estimators to insulate feature distances/margins cleanly
    pipeline = make_pipeline(StandardScaler(), estimator)
    pipeline.fit(Xtr_f, yctr)
    
    # Evaluate predictions against test splits
    predictions = pipeline.predict(Xte_f)
    probabilities = pipeline.predict_proba(Xte_f)[:, 1]
    
    metrics = evaluate_session33_classifier(y_true=ycte, y_pred=predictions, y_probability=probabilities)
    
    session33_models[model_code] = pipeline
    session33_result_rows.append({
        "Model": model_code,
        "Full Model Name": full_model_name,
        "Model Family": model_family,
        "Scaling_Used": True,
        **metrics
    })
    
    print(f"{model_code} Completed - F1 (Class 1): {metrics['f1']:.4f} | At-Risk F1 (Class 0): {metrics['at_risk_f1']:.4f}")

session33_results_df = pd.DataFrame(session33_result_rows).sort_values(by="f1", ascending=False).reset_index(drop=True)
session33_results_df.insert(0, "Session33 F1 Rank", range(1, len(session33_results_df) + 1))

KNN Completed - F1 (Class 1): 0.5455 | At-Risk F1 (Class 0): 0.8246
SVM Completed - F1 (Class 1): 0.7451 | At-Risk F1 (Class 0): 0.8785
NB Completed - F1 (Class 1): 0.7308 | At-Risk F1 (Class 0): 0.8679


In [14]:
classification_metric_columns = [
    "Model", "Full Model Name", "Model Family", "Scaling_Used",
    "accuracy", "precision", "recall", "f1", "roc_auc",
    "at_risk_precision", "at_risk_recall", "at_risk_f1"
]

# Set project paths relative to execution directories
PROJECT_ROOT = Path.cwd().resolve()
for parent in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (parent / ".git").exists():
        PROJECT_ROOT = parent
        break

tables_dir = PROJECT_ROOT / "reports" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
cumulative_path = tables_dir / "classification_model_comparison.csv"

# Slice current session rows to align metrics schema
new_session33_rows = session33_results_df[classification_metric_columns].copy()

if cumulative_path.exists():
    classification_table = pd.read_csv(cumulative_path)
    # Wipe out legacy historical rows of KNN, SVM, or NB to avoid duplicates on re-execution
    if "Model" in classification_table.columns:
        classification_table = classification_table[~classification_table["Model"].isin(["KNN", "SVM", "NB"])].copy()
    
    # Fill missing column dimensions if aligning tables from older updates
    for col in classification_metric_columns:
        if col not in classification_table.columns:
            classification_table[col] = np.nan
            
    classification_table = pd.concat([classification_table[classification_metric_columns], new_session33_rows], ignore_index=True)
else:
    classification_table = new_session33_rows.copy()

# Globally rank across estimators using the baseline F1 metric
classification_table = classification_table.sort_values(by="f1", ascending=False).reset_index(drop=True)
if "Overall F1 Rank" in classification_table.columns:
    classification_table = classification_table.drop(columns=["Overall F1 Rank"])
classification_table.insert(0, "Overall F1 Rank", range(1, len(classification_table) + 1))

# Write updated artifacts to disk
session33_rows_path = tables_dir / "session33_classification_rows.csv"
session33_results_df[classification_metric_columns].to_csv(session33_rows_path, index=False)
classification_table.to_csv(cumulative_path, index=False)

print("Updated Classification Table:")
display(classification_table.style.format({
    "accuracy": "{:.4f}", "precision": "{:.4f}", "recall": "{:.4f}", "f1": "{:.4f}", "roc_auc": "{:.4f}",
    "at_risk_precision": "{:.4f}", "at_risk_recall": "{:.4f}", "at_risk_f1": "{:.4f}"
}))

Updated Classification Table:


,Overall F1 Rank,Model,Full Model Name,Model Family,Scaling_Used,accuracy,precision,recall,f1,roc_auc,at_risk_precision,at_risk_recall,at_risk_f1
0,1,SVM,Support Vector Machine,Maximum-margin,1.000000,0.8354,0.7917,0.7037,0.7451,0.9387,0.8545,0.9038,0.8785
1,2,NB,Gaussian Naive Bayes,Probabilistic,1.000000,0.8228,0.7600,0.7037,0.7308,0.9003,0.8519,0.8846,0.8679
2,3,KNN,K-Nearest Neighbors,Instance-based,1.000000,0.7468,0.7059,0.4444,0.5455,0.7393,0.7581,0.9038,0.8246
3,4,Logistic Regression,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan


### Session 33 Theoretical Interpretation

* **K-Nearest Neighbors (KNN)** assumes that observations close to one another in the scaled feature space tend to share the same target class.
* **Support Vector Machine (SVM)** searches for a structural maximum-margin decision boundary separating output distributions.
* **Gaussian Naive Bayes (NB)** assumes that all predictors are conditionally independent given the class outcome label, modeling continuous predictors using Gaussian tracking parameters within classes.

The Naive Bayes conditional independence assumption is highly unrealistic for student academic performance tracking data. Performance metrics such as earlier period grades, study time, class failures, and historical absenteeism represent tightly interwoven social, behavioral, and academic processes that remain heavily correlated even after conditioning on whether a student passes or falls into the "at-risk" tier. However, it serves as an incredibly lightweight, efficient baseline for our comparison table.

In [15]:
# Automated structural validation checks
expected_models = {"KNN", "SVM", "NB"}
actual_models = set(session33_results_df["Model"])
assert actual_models == expected_models, "Session 33 results missing specific models."
assert len(session33_results_df) == 3, "Result rows mismatch inside standalone table."

# Metrics value bound assertions
required_metrics = ["accuracy", "precision", "recall", "f1", "roc_auc", "at_risk_precision", "at_risk_recall", "at_risk_f1"]
metric_values = session33_results_df[required_metrics].to_numpy(dtype=float)
assert np.isfinite(metric_values).all(), "Non-finite tracking numbers found in results."
assert ((metric_values >= 0) & (metric_values <= 1)).all(), "Metrics bounds exceeded outside [0, 1]."

# Assert that every pipelines contains standard scaling
for code_name, pipeline_obj in session33_models.items():
    assert isinstance(pipeline_obj.steps[0][1], StandardScaler), f"{code_name} pipeline step 0 is not StandardScaler."

print("=" * 72)
print("SESSION 33 GITHUB DELIVERABLE COMPLETED SUCCESSFULLY")
print("=" * 72)

SESSION 33 GITHUB DELIVERABLE COMPLETED SUCCESSFULLY
